In [0]:
from datetime import datetime

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch IdD (1,2 or 3)")

In [0]:
batch_id = dbutils.widgets.get("batch_id")


print("batch_id" , batch_id)


In [0]:
team_name = "team_lemma"
catalog     = f"charles_schwab_retailbrokerage_dev_{team_name}"
landing_volume = f"/Volumes/{catalog}/landing/pwg"
Batch_Folder = f"Batch{batch_id}"

landing_path = f"{landing_volume}/{Batch_Folder}/dailymarket"
Bronze_table = f"{catalog}.bronze.dailymarket"

In [0]:
try:
    run_info_now = (
        spark.read.parquet(landing_path)
             .select("_run_id", "_batch")
             .limit(1)
             .first()
    )
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id

run_id = carried_run_id
print(run_id)

## read from Landing

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
df_landing = spark.read.parquet(landing_path)

print(f"landing count{df_landing.count()}")

df_bronze = df_landing\
                  .drop("_ingest_ts")\
                  .withColumn("_ingest_ts" , current_timestamp())\
                  .withColumn("_batch" , lit(batch_id))\
                  .withColumn("_run_id" , lit(run_id))


print("bronze Schema")
df_bronze.printSchema()



## write to the bronze delta

In [0]:
source_count = df_bronze.count()

df_bronze.write.format("delta").mode("append").partitionBy("_batch").saveAsTable(Bronze_table)

bronze_count = spark.read.table(Bronze_table)\
                        .filter(f"_batch ={batch_id} and _run_id ={run_id}")\
                        .count()


EXPECTED_COUNTS = {"1": 5_270_304, "2": 7_360, "3": 7_360}

expected = EXPECTED_COUNTS[batch_id]

if expected == bronze_count:
    count_status = "pass"
else:
    count_status = "fail"


if source_count == bronze_count:
    match_status = "MAtch"
else:
    match_status = "MisMAtch"


print(f"Source rows  : {source_count}")
print(f"Bronze rows  : {bronze_count}")
print(f"Expected rows: {expected}")
print(f"Match status : {match_status}")
print(f"Count check  : {count_status}")









In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql import Row

recon_results = []

recon_results.append(Row(
        source_table    = "dailymarket",
        batch_id        = Batch_Folder,
        source_count    = source_count,
        target_count    = bronze_count,
        status          = match_status,
        columns_applied = "DM_ACTION,DM_RECID,DM_DATE,DM_S_SYMB,DM_CLOSE,DM_HIGH,DM_LOW,DM_VOL"
    ))

In [0]:
from pyspark.sql import Row


recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if row.status in ("Match" , "Not Match"):
        log_pipeline_recon(
            spark         = spark,
            run_id        = run_id,
            batch_id      = row.batch_id,
            domain        = "MARKET",
            table_name    = row.source_table,
            source_layer  = "landing",
            target_layer  = "bronze",
            source_count  = row.source_count,
            target_count  = row.target_count
        )

        log_audit_event(
            spark = spark,
            run_id = run_id,
            batch = row.batch_id,
            layer = "bronze",
            table_name = row.source_table,
            operation = "Append",
            rows_affected  = row.landing_count
        )

display(recon_df)